In [1]:
import utils.load_data as load_data
import config as cfg
from data_preproc.data_preproc_functions import copy_file, copy_folder, create_folder_if_not_exists, Logger

logger=Logger(None, False)


train_dict, val_dict, test_dict = load_data.get_files(config=cfg, logger=logger)


c:\Users\Daniel (7781D3A3)\.conda\envs\HNC_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-03-08 22:00:54 - INFO: >>> FULL <<<
2024-03-08 22:00:54 - INFO: Expected size (fraction): 1088 (100%)
2024-03-08 22:00:54 - INFO: Size (fraction): 1088 (100.0%)
2024-03-08 22:00:54 - INFO: Group: CT+C_available
2024-03-08 22:00:54 - INFO: 	Value = 0.0, 	Proportion = 17.463% (190)
2024-03-08 22:00:54 - INFO: 	Value = 1.0, 	Proportion = 82.537% (898)
2024-03-08 22:00:54 - INFO: 
2024-03-08 22:00:54 - INFO: Group: CT_Artefact
2024-03-08 22:00:54 - INFO: 	Value = 0.0, 	Proportion = 80.423% (875)
2024-03-08 22:00:54 - INFO: 	Value = 1.0, 	Proportion = 19.577% (213)
2024-03-08 22:00:54 - INFO: 
2024-03-08 22:00:54 - INFO: Group: Photons
2024-03-08 22:00:54 - INFO: 	Value = 0.0, 	Proportion = 13.787% (150)
2024-03-08 22:00

In [5]:
train_dict[0]
import numpy as np

def set_BCE_class_weights(config, train_dict):
    pos_weights = class_ratio_counter(train_dict)
    config.pos_weights = pos_weights
    return config




def class_ratio_counter(train_dict):
    num_endpoints = len(cfg.endpoint_list)
    num_patients =  np.zeros(num_endpoints) # counts the number of patients that have either 0 or 1 for each endpoint (i.e. that are not missing data)
    class_counts = np.zeros(num_endpoints)  # counts the number of patients that are equal to 1 (i.e. have the toxicity)
    for patient in train_dict:
        labels = np.array(patient['label_list'])
        data_present = labels>=0
        num_patients += data_present
        labels[labels<0]=0
        class_counts += labels

    print(class_counts)
    print(num_patients)
    pos_weights = num_patients/(2*class_counts)
    inv_class_counts = num_patients - class_counts

    neg_weights = num_patients/(2*inv_class_counts)


    return inv_class_counts/class_counts


class_ratio_counter(train_dict)

[ 45. 156. 193. 148. 255.]
[552. 592. 555. 555. 558.]


array([11.26666667,  2.79487179,  1.87564767,  2.75      ,  1.18823529])

In [14]:
train_dict[0]

{'ct': "C:/Users/danie/Desktop/FSE 23-34/Master's Project/DL_NTCP_Multitox\\datasets\\dataset_dan\\patients\\0005680\\ct.npy",
 'rtdose': "C:/Users/danie/Desktop/FSE 23-34/Master's Project/DL_NTCP_Multitox\\datasets\\dataset_dan\\patients\\0005680\\rtdose.npy",
 'segmentation_map': "C:/Users/danie/Desktop/FSE 23-34/Master's Project/DL_NTCP_Multitox\\datasets\\dataset_dan\\patients\\0005680\\segmentation_map.npy",
 'features': [1.0,
  0.68,
  1.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  1.0,
  0.0,
  0.0,
  0.0],
 'label_list': [0, 0, 1, 0, 0],
 'patient_id': '0005680'}

In [5]:
(train_dl, val_dl, test_dl, batch_size, channels, depth, height, width, 
            n_features, train_dict, val_dict, test_dict) = \
            load_data.main(config=cfg, train_dict=train_dict, val_dict=val_dict, test_dict=test_dict,
                           fold=0, drop_last=cfg.dataloader_drop_last, logger=logger)

2024-03-03 19:08:24 - INFO: Extra features for Deep Learning model: ['Sex', 'Age', 'Xerostomia_W01_Helemaal_niet', 'Xerostomia_W01_Een_beetje', 'Xerostomia_W01_Nogal_Heel_erg', 'Aspiration_W01_Helemaal_niet', 'Aspiration_W01_Een_beetje', 'Aspiration_W01_Nogal_Heel_erg', 'Sticky_W01_Helemaal_niet', 'Sticky_W01_Een_beetje', 'Sticky_W01_Nogal_Heel_erg', 'Taste_W01_Helemaal_niet', 'Taste_W01_Een_beetje', 'Taste_W01_Nogal_Heel_erg', 'Dysphagia_W01_Grade0_1', 'Dysphagia_W01_Grade2', 'Dysphagia_W01_Grade3_4', 'Loctum2_Larynx', 'Loctum2_Oral_Cavity', 'Loctum2_Overig', 'Loctum2_Pharynx'].
2024-03-03 19:08:24 - INFO: Total number of data samples: 1088.
2024-03-03 19:08:24 - INFO: Endpoint: Xerostomia_M06
2024-03-03 19:08:24 - INFO: 	Training size (labels=0): 303/610 (0.497).
2024-03-03 19:08:24 - INFO: 	Training size (labels=1): 255/610 (0.418).
2024-03-03 19:08:24 - INFO: 	Internal validation size (labels=0): 137/261 (0.525).
2024-03-03 19:08:24 - INFO: 	Internal validation size (labels=1): 92/

In [8]:
train_dl.__dict__

{'dataset': <monai.data.dataset.CacheDataset at 0x1bfaa856bf0>,
 'num_workers': 8,
 'prefetch_factor': 2,
 'pin_memory': True,
 'pin_memory_device': '',
 'timeout': 0,
 'worker_init_fn': <function monai.data.utils.worker_init_fn(worker_id: 'int') -> 'None'>,
 '_DataLoader__multiprocessing_context': None,
 '_dataset_kind': 0,
 'batch_size': 8,
 'drop_last': True,
 'sampler': <torch.utils.data.sampler.RandomSampler at 0x1bfaaebee00>,
 'batch_sampler': <torch.utils.data.sampler.BatchSampler at 0x1bfaaebefb0>,
 'generator': None,
 'collate_fn': <function monai.data.utils.list_data_collate(batch: 'Sequence')>,
 'persistent_workers': True,
 '_DataLoader__initialized': True,
 '_IterableDataset_len_called': None,
 '_iterator': <torch.utils.data.dataloader._MultiProcessingDataLoaderIter at 0x1bfaa6d9ab0>}

In [ ]:
for i, batch_data in enumerate(val_dataloader):
    pass

In [ ]:
batch_data

{'features': tensor([[0.0000, 0.6300, 1.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 1.0000,
          0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 1.0000, 1.0000,
          0.0000, 0.0000, 0.0000]]),
 'label_list': tensor([[1, 0, 1, 1, 0]]),
 'patient_id': ['7208529'],
 'ct_dose_seg': tensor([[[[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
            [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
            [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
            ...,
            [1.0000, 0.9167, 0.5650,  ..., 0.4283, 0.4683, 0.6233],
            [0.5350, 0.8800, 1.0000,  ..., 0.6117, 0.8183, 1.0000],
            [0.4617, 0.4617, 0.4833,  ..., 0.9717, 1.0000, 0.6933]],
 
           [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
            [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
            [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
            ...,
            [0.7300, 1.0000, 1.0000,  ..., 0.5817, 0.

In [4]:
import time
import tracemalloc
import misc

start = time.time()

tracemalloc.start()
for i, batch_data in enumerate(val_dataloader):

    #batch_data = batch
    #print(batch_data['ct_dose_seg'].device)
    train_inputs, train_features, train_label_list = (
                batch_data['ct_dose_seg'].to(cfg.device),
                batch_data['features'].to(cfg.device),
                batch_data['label_list'].to(cfg.device)
            )

    misc.plot_model_inputs(config=cfg, plot_inputs=train_inputs, mean=norm_mean, std=norm_std, epoch=i)

end = time.time()
print(end - start)

# displaying the memory
print(tracemalloc.get_traced_memory())
 
# stopping the library
tracemalloc.stop()





165.01338481903076
(30345295, 68953269)


In [5]:
len(train_dataloader)

2

In [5]:
cfg.device

device(type='cuda')

In [4]:
batch_data = batch
train_inputs, train_features, train_label_list = (
                batch_data['ct_dose_seg'].to(cfg.device),
                batch_data['features'].to(cfg.device),
                batch_data['label_list'].to(cfg.device)
            )

In [8]:
import process_data

for b in range(len(train_inputs)):
                        # Generate a random 32-bit integer
    seed_b = 0
    train_inputs[b] = process_data.aug_mix(arr=train_inputs[b], mixture_width=cfg.mixture_width,
                                        mixture_depth=cfg.mixture_depth, augmix_strength=cfg.augmix_strength,
                                        device=cfg.device, seed=seed_b)
    
train_inputs = process_data.preprocess_inputs(train_inputs, norm_mean, norm_std, rtdose_mean=0, rtdose_std=1)


In [7]:
import misc

misc.plot_model_inputs(config=cfg, plot_inputs=train_inputs, mean=norm_mean, std=norm_std, epoch=0)

In [6]:
import data_preproc_config
import os
import misc

def plot_model_inputs2(config, plot_inputs, mean, std, epoch):
    """
    Plots the model inputs (CT, RTDOSE, Segmentation) used during training, and saves the figure as a png.
    Args:
        plot_inputs (list): list of model input images
        mean (list): list of means, one value for each channel (first value for CT, second value for RTDOSE, etc)
        std (list): list of stds, one value for each channel (same as mean)
        epoch (int): current epoch number
    """
    
    for patient_idx in range(min(config.max_nr_images_per_interval, len(plot_inputs))):
        # rescale the arrays back to the original (a_max to a_min) scales
        # eg:   (plot_inputs[CT] * (400 - (-200))) + (-200)
        ct_range = (config.ct_a_max - config.ct_a_min)
        rtdose_range = (config.rtdose_a_max - config.rtdose_a_min)
        arr_list = [    (plot_inputs[patient_idx][0].cpu() * ct_range) + config.ct_a_min,
                        (plot_inputs[patient_idx][1].cpu() * rtdose_range) + config.rtdose_a_min,
                        plot_inputs[patient_idx][2].cpu()]
        cmap_list = [data_preproc_config.ct_cmap, data_preproc_config.rtdose_cmap, data_preproc_config.segmentation_cmap]
        # vmin_list = [0, 0, 0]
        vmin_list = [config.ct_a_min, config.rtdose_a_min, 0]
        # vmax_list = [1, config.data_preproc_config.rtdose_vmax/config.rtdose_a_max, 1]
        vmax_list = [config.ct_a_max, config.rtdose_a_max, 1]
        ticks_steps_list = [vmax_list[0] - vmin_list[0], vmax_list[1] - vmin_list[1], 1]
        segmentation_map_list = [arr_list[2] >= 0.5, None, None]
        colorbar_title_list = [data_preproc_config.ct_colorbar_title,
                                           data_preproc_config.rtdose_colorbar_title,
                                           data_preproc_config.segmentation_colorbar_title]
        # Overlay
        overlay_list = [None, None, None]
        alpha_overlay_list = [None, None, None]
        cmap_overlay_list = [None, None, None]
        vmin_overlay_list = [None, None, None]
        vmax_overlay_list = [None, None, None]
        misc.plot_multiple_arrs(arr_list=arr_list, overlay_list=overlay_list,
                                       alpha_overlay_list=alpha_overlay_list,
                                       nr_images=data_preproc_config.plot_nr_images_multiple,
                                       figsize=data_preproc_config.figsize_multiple, cmap_list=cmap_list,
                                       cmap_overlay_list=cmap_overlay_list, colorbar_title_list=colorbar_title_list,
                                       filename=os.path.join(config.exp_figures_dir,
                                                             'epoch_{}_idx_{}.png'.format(epoch + 1, patient_idx)),
                                       vmin_list=vmin_list, vmin_overlay_list=vmin_overlay_list,
                                       vmax_list=vmax_list,
                                       vmax_overlay_list=vmax_overlay_list, ticks_steps_list=ticks_steps_list,
                                       segmentation_map_list=segmentation_map_list)

plot_model_inputs2(config=cfg, plot_inputs=train_inputs, mean=norm_mean, std=norm_std, epoch=0)

In [ ]:
original_array = (normalized_array * (400 - (-200))) + (-200)


In [27]:
import process_data

start = time.time()

tracemalloc.start()
for i in range(100):
    train_inputs = process_data.preprocess_inputs(inputs=train_inputs, ct_mean=norm_mean[0], ct_std=norm_std[0],
                                                            rtdose_mean=norm_mean[1], rtdose_std=norm_std[1])

print(tracemalloc.get_traced_memory())
 
# stopping the library
tracemalloc.stop()

print(time.time() - start)

(12729, 23023)
0.2588932514190674


In [6]:
norm_mean

[0.20998462, 0.14687346, 0.01263298]

In [28]:
import random
import time
import tracemalloc

start_aug = time.time()
tracemalloc.start()
for i in range(10):
    for b in range(len(train_inputs)):
        # Generate a random 32-bit integer
        seed_b = random.getrandbits(32)
        train_inputs[b] = process_data.aug_mix(arr=train_inputs[b], mixture_width=cfg.mixture_width,
                                            mixture_depth=cfg.mixture_depth, augmix_strength=cfg.augmix_strength,
                                            device=cfg.device, seed=seed_b)
print(tracemalloc.get_traced_memory())

# stopping the library
tracemalloc.stop()

print(f'Augmix time: {time.time()-start_aug:.2f} seconds.')

(336015, 683217)
Augmix time: 18.88 seconds.


In [9]:
start = time.time()

for batch in train_dataloader:
    pass

end = time.time()
print(end - start)

16.74290704727173


ValueError: dataset attribute should not be set after DataLoader is initialized

In [3]:
train_dataloader.dataset

In [10]:
dataset = train_dataloader.dataset

In [11]:
dataset

In [18]:
dataset.shape

AttributeError: 'PersistentDataset' object has no attribute 'shape'

In [35]:
from monai.transforms import (
    Transform,
    Compose,
    ConcatItemsd,
    CenterSpatialCropd,
    DeleteItemsd,
    LoadImaged,
    EnsureChannelFirstd,
    EnsureTyped,
    NormalizeIntensityd,
    Resized,
    RandSpatialCropd,
    Rand3DElasticd,
    RandAdjustContrastd,
    RandAffined,
    RandFlipd,
    RandGaussianNoised,
    RandGridDistortiond,
    RandRotated,
    RandZoomd,
    RandShiftIntensityd,
    ScaleIntensityRanged,
    ScaleIntensityRange,
    MapLabelValued,
    ToDeviced,
    CopyItemsd,
)
import numpy as np

In [44]:
from process_data import aug_list
import torch
import config

class AugMix(Transform):
    def __init__(self, keys, config):
        self.keys = keys
        self.mixture_width = config.mixture_width
        self.mixture_depth = config.mixture_depth
        self.augmix_strength = config.augmix_strength
        self.device = config.device
        self.seed = config.seed

        self.ct_interpol_mode_2d = config.ct_interpol_mode_2d
        self.rtdose_interpol_mode_2d = config.rtdose_interpol_mode_2d
        self.segmentation_interpol_mode_2d = config.segmentation_interpol_mode_2d

    def __call__(self, input):
        # Generate a random 32-bit integer
        ws = torch.tensor(np.random.dirichlet([1] * self.mixture_width), dtype=torch.float32)
        m = torch.tensor(np.random.beta(1, 1), dtype=torch.float32)
        # m = np.random.uniform(0, 0.25)

        mix = torch.zeros_like(input, device=self.device)
        for i in range(self.mixture_width):
            image_aug = input.clone()

            # OLD
            # depth = mixture_depth if mixture_depth > 0 else np.random.randint(1, 4)
            depth = np.random.randint(self.mixture_depth[0], self.mixture_depth[1] + 1)
            for _ in range(depth):
                # op = np.random.choice(aug_list)
                idx = random.randint(0, len(aug_list) - 1)
                op = aug_list[idx]
                seed_i = random.getrandbits(32)

                # CT
                image_aug[0] = op(arr=image_aug[0], mode=self.ct_interpol_mode_2d, strength=self.augmix_strength, seed=seed_i)
                # RTDOSE
                image_aug[1] = op(arr=image_aug[1], mode=self.rtdose_interpol_mode_2d, strength=self.augmix_strength,
                                seed=seed_i)
                # Segmentation
                image_aug[2] = op(arr=image_aug[2], mode=self.segmentation_interpol_mode_2d, strength=self.augmix_strength,
                                seed=seed_i)

            # Preprocessing commutes since all coefficients are convex
            mix += ws[i] * image_aug

        mixed = (1 - m) * input + m * mix

        return mixed
    

class NormaliseInputs(Transform):
    def __init__(self, keys, config):
        self.keys = keys
        self.config = config

        self.ct_scaler = ScaleIntensityRange(a_min=config.ct_a_min, a_max=config.ct_a_max,
                                    b_min=config.ct_b_min, b_max=config.ct_b_max,
                                    clip=config.ct_clip)
        self.rtdose_scaler = ScaleIntensityRange(a_min=config.rtdose_a_min, a_max=config.rtdose_a_max,
                                        b_min=config.rtdose_b_min, b_max=config.rtdose_b_max,
                                        clip=config.rtdose_clip)
        
        
    def __call__(self, input):
        input[:, 0, ...] = self.ct_scaler(input[:, 0, ...])
        input[:, 1, ...] = self.rtdose_scaler(input[:, 1, ...])
        return input

In [45]:
def get_transforms(config, logger):
    """
    Transforms for training, internal validation and test data.

    Source: https://github.com/Project-MONAI/tutorials/blob/master/modules/3d_image_transforms.ipynb
    Source: https://docs.monai.io/en/latest/transforms.html?highlight=Rand3DElastic#monai.transforms.Rand3DElastic

    Note: MONAI documentation describes that the transformers require 3D data to have shape (nchannels, H, W, D), but
        shapes (1, H, W, D) and (H, W, D) results in same augmented data, i.e. shape (H, W, D) is fine.

    LoadImaged: https://docs.monai.io/en/latest/transforms.html?highlight=loadimaged#monai.transforms.LoadImage
    EnsureTyped: https://docs.monai.io/en/latest/transforms.html?highlight=ensuretyped#monai.transforms.EnsureType
    Resized: https://docs.monai.io/en/latest/transforms.html?highlight=resized#monai.transforms.Resize
    ToDeviced: Move PyTorch Tensor to the specified device. It can help cache data into GPU and execute on GPU directly.
        CacheDataset caches the transform results until ToDeviced, so it is in GPU memory. Then in every epoch, the
        program fetches cached data from GPU memory and only execute the transforms after `ToDeviced` on GPU directly.
    Source: https://github.com/Project-MONAI/tutorials/blob/main/acceleration/fast_model_training_guide.md
    Source: https://docs.monai.io/en/latest/transforms.html?highlight=todeviced#monai.transforms.ToDevice

    Args:
        config:
        logger:

    Returns:

    """

    perform_data_aug = config.perform_data_aug
    data_aug_strength = config.data_aug_strength
    data_aug_p = config.data_aug_p
    
    rand_cropping_size = config.rand_cropping_size
    input_size = config.input_size
    seg_orig_labels = config.seg_orig_labels
    seg_target_labels = config.seg_target_labels

    device = config.device
    to_device = config.to_device
    image_keys = config.image_keys
    concat_key = config.concat_key
    # Define variables for exceptions

    modes_3d = [config.ct_dose_seg_interpol_mode_3d]
    modes_2d = [config.ct_dose_seg_interpol_mode_2d]
    align_corners_exception_3d = [True if mode in ['linear', 'bilinear', 'bicubic', 'trilinear'] else None for mode in
                                  modes_3d]
    align_corners_exception_2d = [True if mode in ['linear', 'bilinear', 'bicubic', 'trilinear'] else None for mode in
                                  modes_2d]
    # CT
    range_value_ct = config.ct_a_max - config.ct_a_min
    # RTDOSE
    range_value_rtdose = config.rtdose_a_max - config.rtdose_a_min
    # Segmentation_map
    # values_filter = [data_preproc_config.structures_values[x] for x in config.segmentation_structures]

    logger.my_print('To_device: {}.'.format(to_device))

    # Define generic transforms
    generic_transforms = Compose([
        LoadImaged(keys=image_keys),
        EnsureTyped(keys=image_keys + ['features', 'label_list'], data_type='tensor'),
        # Clip
        ScaleIntensityRanged(keys=['ct'],
                             a_min=config.ct_a_min, a_max=config.ct_a_max,
                             b_min=config.ct_a_min, b_max=config.ct_a_max,
                             clip=config.ct_clip),
        ScaleIntensityRanged(keys=['rtdose'],
                             a_min=config.rtdose_a_min, a_max=config.rtdose_a_max,
                             b_min=config.rtdose_a_min, b_max=config.rtdose_a_max,
                             clip=config.rtdose_clip),
        # Useful for creating figures such as attention maps
        # CopyItemsd(keys=['segmentation_map'], names=['segmentation_map_original'], times=1),
        MapLabelValued(keys=['segmentation_map'], orig_labels=seg_orig_labels, target_labels=seg_target_labels),
        ConcatItemsd(keys=image_keys, name=concat_key, dim=0),
        DeleteItemsd(keys=image_keys),
    ])

    # Define training transforms
    train_transforms = generic_transforms

    # Define internal validation and test transforms
    val_transforms = generic_transforms

    # Data augmentation
    if perform_data_aug:
        train_transforms = Compose([
            train_transforms,
            RandSpatialCropd(keys=[concat_key], roi_size=rand_cropping_size, random_center=True, random_size=False),
            RandFlipd(keys=[concat_key], prob=data_aug_p, spatial_axis=-1),  # 3D: (num_channels, H[, W, …, ])
            RandAffined(keys=[concat_key], prob=data_aug_p,
                        translate_range=(7 * data_aug_strength, 7 * data_aug_strength, 7 * data_aug_strength),
                        padding_mode='border', mode=modes_2d),  # 3D: (num_channels, H, W[, D])
            RandAffined(keys=[concat_key], prob=data_aug_p,
                        scale_range=(0.07 * data_aug_strength, 0.07 * data_aug_strength, 0.07 * data_aug_strength),
                        padding_mode='border', mode=modes_2d),  # 3D: (num_channels, H, W[, D])
            # RandZoomd(keys=concat_key, prob=data_aug_p, min_zoom=[1] + [1 - 0.07 * data_aug_strength] * 2,
            #           max_zoom=[1] + [1 + 0.07 * data_aug_strength] * 2, align_corners=align_corners_exception_3d,
            #           padding_mode='edge', mode=modes_3d),  # Is similar to  RandAffined with scale_range param.
            # 3D: (nchannels, H, W, D). Only `trilinear` possible for 5D input data
            RandRotated(keys=[concat_key], prob=data_aug_p, range_x=(np.pi / 24) * data_aug_strength,
                        align_corners=align_corners_exception_2d, padding_mode='border', mode=modes_2d),
            # 3D: (nchannels, H, W, D). Only `bicubic`, `nearest` and `bilinear` possible. `Trilinear` not available because
            # the rotation is done in 2 dimensions, so that every slice is rotated by the same amount.
            # Rand3DElasticd(keys=[concat_key], prob=data_aug_p, sigma_range=(5, 8), magnitude_range=(0, round(1 * data_aug_strength)),
            #                padding_mode='border', mode=modes_3d),  # 3D: (nchannels, H, W, D)
            RandShiftIntensityd(keys=[concat_key], prob=data_aug_p,
                                offsets=(0, 0.05 * data_aug_strength), 
                                channel_wise=True
                                ),
            # RandGaussianNoised(keys=['ct'], prob=data_aug_p, mean=0.0, std=(0.1 / (range_value_ct ** (1 / 2))) * data_aug_strength),
            # RandGaussianNoised(keys=['rtdose'], prob=data_aug_p, mean=0.0, std=(0.1 / (range_value_rtdose ** (1 / 2))) * data_aug_strength),
        ])

    # Resize images
    if list(rand_cropping_size) != list(input_size) or not perform_data_aug:
        train_transforms = Compose([
            train_transforms,
            CenterSpatialCropd(keys=[concat_key], roi_size=input_size),
        ])

    val_transforms = Compose([
        val_transforms,
        CenterSpatialCropd(keys=[concat_key], roi_size=input_size),
    ])

    # To device
    if to_device:
        train_transforms = Compose([
            train_transforms,
            ToDeviced(keys=[concat_key], device=device),
        ])

        val_transforms = Compose([
            val_transforms,
            ToDeviced(keys=[concat_key], device=device),
        ])
    
    if config.perform_augmix:
        train_transforms = Compose([
            train_transforms,
            AugMix(keys=[concat_key], config=config)
        ])
    
    # Finally, a normalisation transform step (to normalise the model's input images to [0,1])
    train_transforms = Compose([
        train_transforms,
        NormaliseInputs(keys=[concat_key], config=config)
    ])
    val_transforms = Compose([
        val_transforms,
        NormaliseInputs(keys=[concat_key], config=config)
    ])

    # Flatten transforms
    train_transforms = train_transforms.flatten()
    val_transforms = val_transforms.flatten()

    # Print transforms
    for mode, t in zip(['Train', 'Validation'], [train_transforms, val_transforms]):
        logger.my_print('{} transforms:'.format(mode))
        for i in t.transforms:
            logger.my_print('\t{}, keys={}'.format(i.__class__, i.keys))

    return train_transforms, val_transforms


train_transforms, val_transforms = get_transforms(cfg, logger)


2024-02-22 21:41:16,906 - INFO: To_device: False.


NameError: name 'ScaleIntensityRange' is not defined